# Prepare z-Only Synthetic Observations

This notebook prepares the observation dataset for the first information-channel experiment proposed by Takemasa: **observe only $z$ while keeping the later CSE interventions restricted to $z$**.

To make the comparison with the existing full-state experiment as controlled as possible, no new noise realization is generated. The $z$ row is extracted from the already validated full-state observation file produced by `2_gen_obs.ipynb`. Therefore, the observation times, truth trajectory, error variance, and actual noise samples in $z$ are unchanged; only $x$ and $y$ are withheld from data assimilation.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from class_obs import obs_da
from class_state_vec import state_vector
from project_paths import NATURE_DATA_DIR, OBSERVATIONS_DATA_DIR, OBSERVATIONS_FIGURES_DIR

OBSERVED_INDICES = (2,)
OBSERVATION_LABEL = 'z_only_obs'

OUTPUT_ROOT = OBSERVATIONS_DATA_DIR.parent / OBSERVATION_LABEL
DATA_DIR = OUTPUT_ROOT / 'data'
FIGURES_DIR = OUTPUT_ROOT / 'figures'
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

NATURE_FILE = NATURE_DATA_DIR / 'lorenz63_run.pkl'
FULL_OBSERVATION_FILE = OBSERVATIONS_DATA_DIR / 'y_obs.pkl'
OUTPUT_FILE = DATA_DIR / 'y_obs_z_only.pkl'

print('Full-state input:', FULL_OBSERVATION_FILE)
print('z-only output:', OUTPUT_FILE)


## Extract the Existing z Observation Channel

The selector is

\[
H_z = \begin{bmatrix}0 & 0 & 1\end{bmatrix}.
\]

The saved observation object has shape $(1,N)$ and position index 2 at every time. This is a new artifact in an isolated directory; `y_obs.pkl` is never modified.


In [ ]:
sv = state_vector().load(NATURE_FILE)
x_nature = np.asarray(sv.getTrajectory(), dtype=float)

full_obs = obs_da().load(FULL_OBSERVATION_FILE)
full_values = np.asarray(full_obs.getVal(), dtype=float)
full_positions = np.asarray(full_obs.getPos(), dtype=int)

if full_values.shape != x_nature.shape:
    raise ValueError(
        f'Full observations {full_values.shape} and nature run {x_nature.shape} are not aligned.'
    )
if full_positions.shape != full_values.shape:
    raise ValueError('Full observation positions do not match the observation array.')

z_values = full_values[np.asarray(OBSERVED_INDICES), :].copy()
z_positions = np.full(z_values.shape, OBSERVED_INDICES[0], dtype=int)

z_obs = obs_da(name='observe_z_only')
z_obs.setVal(z_values)
z_obs.setPos(z_positions)
z_obs.setErr(np.array([1.5], dtype=float))
z_obs.observed_indices = list(OBSERVED_INDICES)
z_obs.source_observation_file = str(FULL_OBSERVATION_FILE)
z_obs.noise_variance = 1.5
z_obs.common_random_numbers = True
z_obs.save(OUTPUT_FILE)

assert z_values.shape == (1, x_nature.shape[1])
assert np.array_equal(z_values[0], full_values[2])
assert np.all(z_positions == 2)

print('Nature-run shape:', x_nature.shape)
print('z-only observation shape:', z_values.shape)
print('The saved z values exactly match row 2 of the full-state observations.')


## Visual Verification


In [ ]:
window = 500
steps = np.arange(window)

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(steps, x_nature[2, :window], color='black', linewidth=1.3, label='nature z')
ax.scatter(steps, z_values[0, :window], s=9, color='tab:purple', alpha=0.55, label='observed z')
ax.set_xlabel('Time step')
ax.set_ylabel('z')
ax.set_title('z-only synthetic observations: unchanged z noise realization')
ax.grid(alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
figure_path = FIGURES_DIR / 'z_only_observations_first_500_steps.png'
fig.savefig(figure_path, dpi=300, bbox_inches='tight')
plt.show()

reloaded = obs_da().load(OUTPUT_FILE)
assert np.array_equal(reloaded.getVal(), z_values)
assert np.array_equal(reloaded.getPos(), z_positions)
print('Reload validation passed:', OUTPUT_FILE)
print('Figure:', figure_path)
